# Chương 4: Mạng Nơ-ron Hồi Quy (RNN) & Dự Báo Chuỗi Thời Gian
**Học viện Công nghệ Bưu chính Viễn thông (PTIT)**
*Môn học: Phát triển các Hệ thống Thông minh (GVHD: PGS.TS Trần Đình Quế)*
*Sinh viên: Phạm Văn Tư - B23DCCN879*

## 1. Tế Bào LSTM Scratch Bằng NumPy
Hiện thực hóa phương trình dòng truyền thông tin qua 3 cổng Forget, Input, Output và Cell State:

In [1]:
import numpy as np

class LSTMCellScratch:
    def __init__(self, input_dim, hidden_dim):
        self.W = np.random.randn(input_dim + hidden_dim, 4 * hidden_dim) * 0.05
        self.b = np.zeros((1, 4 * hidden_dim))
        self.hidden_dim = hidden_dim

    def forward(self, x_t, h_prev, c_prev):
        concat = np.hstack((h_prev, x_t))
        gates = np.dot(concat, self.W) + self.b
        f = 1.0 / (1.0 + np.exp(-gates[:, :self.hidden_dim]))
        i = 1.0 / (1.0 + np.exp(-gates[:, self.hidden_dim:2*self.hidden_dim]))
        c_cand = np.tanh(gates[:, 2*self.hidden_dim:3*self.hidden_dim])
        o = 1.0 / (1.0 + np.exp(-gates[:, 3*self.hidden_dim:]))
        c_t = f * c_prev + i * c_cand
        h_t = o * np.tanh(c_t)
        return h_t, c_t

cell = LSTMCellScratch(input_dim=5, hidden_dim=16)
h0 = np.zeros((1, 16))
c0 = np.zeros((1, 16))
x0 = np.random.randn(1, 5)
h1, c1 = cell.forward(x0, h0, c0)
print("Kích thước Hidden State h1:", h1.shape)


Kích thước Hidden State h1: (1, 16)


## 2. Huấn Luyện BiLSTM Dự Báo Cổ Phiếu FPT.VN Bằng PyTorch:

In [2]:
import torch
import torch.nn as nn

class BiLSTMPredictor(nn.Module):
    def __init__(self, input_dim=5, hidden_dim=64, num_layers=2, output_dim=1):
        super(BiLSTMPredictor, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        out, _ = self.lstm(x)
        last_step = out[:, -1, :]
        return self.fc(last_step)

model_ts = BiLSTMPredictor()
dummy_series = torch.randn(16, 30, 5) # 16 batches, 30 ngày cửa sổ, 5 đặc trưng
preds = model_ts(dummy_series)
print("Dự báo giá ngày tiếp theo:", preds.shape)


Dự báo giá ngày tiếp theo: torch.Size([16, 1])
